# Pytorch Bridge

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dujing82-blip/usaaio_numpy_course/blob/main/notebooks/07_pytorch_bridge.ipynb)

**English • CPU • synthetic data • predict shapes before coding**

Run Setup, then work down the page. Edit only YOUR CODE HERE. A blank answer prints FAIL; that is normal. Complete every component until the grader says ALL PASS.

In [ ]:
# @title Setup (run first)
!pip -q install git+https://github.com/dujing82-blip/usaaio_numpy_course.git
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from usaaio_check import grade, check_rules, show_shape, NO_LOOPS, PURE
from usaaio_check.exercise_tests import evaluate, regression_args, mlp_args
from usaaio_check.datasets import random_embeddings, blobs, titanic_like, iaio_kmeans_table
np.set_printoptions(precision=4, suppress=True)
exercise_results = {}
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
torch.set_num_threads(1)
torch.manual_seed(2026)


## Before you start

PyTorch tensors are arrays that can also remember how they were computed. `requires_grad=True` starts a computation graph; `backward` or `autograd.grad` follows it to calculate derivatives. `detach()` disconnects a tensor from the graph; `item()` extracts one scalar. `torch.no_grad()` is useful for evaluation or copying parameters.

| NumPy | PyTorch |
|---|---|
| axis, keepdims | dim, keepdim |
| expand_dims | unsqueeze |
| transpose with axes | permute |
| concatenate | cat |
| ndarray | Tensor |

All tasks here run on CPU. You can explicitly use `tensor.to('cpu')`. `torch.from_numpy` can share memory; `torch.tensor` creates a copy.

## Vocabulary pit stop

`view` reshapes a contiguous tensor; `reshape` can also copy when needed. `from_numpy` can share storage. A 2-D grid is built by pairing two coordinate axes.

In [ ]:
a = np.array([1.,2.,3.],dtype=np.float32)
t = torch.from_numpy(a).float().to('cpu')
show_shape(t=t, column=t.view(-1,1))
u = torch.linspace(0,1,3)
g0,g1 = torch.meshgrid(u,u,indexing='ij')
show_shape(grid0=g0, grid1=g1)
print(t.detach().numpy())

### Exercise 7.1 — NumPy ↔ torch vocabulary ★★★

[A4] [A5] [A6] [B5] [C4] [E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** redo 1.8, 1.9, 1.12, 2.8, 3.9 with `torch` (`unsqueeze`, `squeeze`, `permute`, `cat`, `stack`, `dim=`, `keepdim=`)

**Shape contract:** same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW

**Rules:** torch_regroup: no-loops; torch_shape_walk: no-loops; torch_join: no-loops; torch_unit_rows: no-loops; torch_sim_matrix: no-loops; torch_to_chw: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW.

Add a length-one axis with unsqueeze. from_numpy shares memory; use clone when you need a separate tensor.

**My prediction:** _Write your answer here before coding._

In [ ]:
def torch_regroup(a):
    'torch_regroup: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    return a.reshape(2,3),a.reshape(-1,1),a.flatten()
def torch_shape_walk(arr):
    'torch_shape_walk: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    a = arr.squeeze()
    b = a.unsqueeze(2)
    c = b.transpose(0,1)
    return tuple(a.shape),tuple(b.shape),tuple(c.shape),tuple(c.flatten().shape)
def torch_join(a,b):
    'torch_join: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    return torch.cat([a,b]),torch.stack([a,b]),torch.stack([a,b],dim=1)
def torch_unit_rows(V):
    'torch_unit_rows: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    return V/torch.norm(V,dim=-1,keepdim=True)
def torch_sim_matrix(W):
    'torch_sim_matrix: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    return torch.sum(W[:,None,:]*W[None,:,:],dim=-1)
def torch_to_chw(img):
    'torch_to_chw: same shapes as NumPy exercises 1.8, 1.9, 1.12, 2.8, 3.9, and HWC -> CHW. See the exercise task for the required behavior.'
    return img.permute(2,0,1)


In [ ]:
# @title Grade 7.1 (run, do not edit)
exercise_results['7.1'] = evaluate(globals(), '7.1')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

PyTorch uses dim and keepdim where NumPy uses axis and keepdims.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 7.2 — `My_ReLU`, `My_Threshold`, `My_Linear` as `nn.Module` (2025 R1 P2.10, 2026 R1 P7.1–7.3) ★★★

[B1] [E1] [E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** `torch.max(torch.zeros_like(x), x)`; `(x>=0).float()`; `nn.Parameter(torch.randn(out,in)*std)` with `std = 1/sqrt(in)` for unit output variance (2026 R1 P7.2), zero bias, `bias=False` option

**Shape contract:** activations: same shape; linear x: (...,in), output: (...,out)

**Rules:** My_ReLU.forward: no-loops; My_Threshold.forward: no-loops; My_Linear.forward: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: activations: same shape; linear x: (...,in), output: (...,out).

**My prediction:** _Write your answer here before coding._

In [ ]:
class My_ReLU(nn.Module):
    def forward(self,x):
        'forward: activations: same shape; linear x: (...,in), output: (...,out). See the exercise task for the required behavior.'
        return torch.max(torch.zeros_like(x),x)
class My_Threshold(nn.Module):
    def forward(self,x):
        'forward: activations: same shape; linear x: (...,in), output: (...,out). See the exercise task for the required behavior.'
        return (x>=0).float()
class My_Linear(nn.Module):
    def __init__(self,in_features,out_features,bias=True):
        '__init__: activations: same shape; linear x: (...,in), output: (...,out). See the exercise task for the required behavior.'
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features,in_features)/np.sqrt(in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
    def forward(self,x):
        'forward: activations: same shape; linear x: (...,in), output: (...,out). See the exercise task for the required behavior.'
        out = x@self.weight.T
        return out+self.bias if self.bias is not None else out


In [ ]:
# @title Grade 7.2 (run, do not edit)
exercise_results['7.2'] = evaluate(globals(), '7.2')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Register trainable tensors with nn.Parameter; initialize weights with std = 1/sqrt(in).

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 7.3 — `My_MLP_Model` with `nn.Sequential` + training loop on `y = sin(2πx) + noise` (2025 R1 P2.11–2.13) ★★★

[E1] [E2] [E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** MSE, Adam, whole batch, print every 10 epochs with 4 decimals, loss plot

**Shape contract:** x,y: (N,1); outputs: trained module and loss history (epochs,)

**Rules:** train_mlp: at most 1 explicit iteration loop; My_MLP_Model.forward: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: x,y: (N,1); outputs: trained module and loss history (epochs,).

Whole-batch CPU training; print every 10 epochs to four decimal places.

**My prediction:** _Write your answer here before coding._

In [ ]:
class My_MLP_Model(nn.Module):
    def __init__(self):
        '__init__: x,y: (N,1); outputs: trained module and loss history (epochs,). See the exercise task for the required behavior.'
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,1))
    def forward(self,x):
        'forward: x,y: (N,1); outputs: trained module and loss history (epochs,). See the exercise task for the required behavior.'
        return self.layers(x)
def train_mlp(x,y,epochs=80):
    'train_mlp: x,y: (N,1); outputs: trained module and loss history (epochs,). See the exercise task for the required behavior.'
    torch.manual_seed(42)
    model = My_MLP_Model()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.03)
    history = np.zeros(epochs)
    for epoch in range(epochs):
        optimizer.zero_grad()
        prediction = model(x)
        loss = torch.mean((prediction-y)**2)
        loss.backward()
        optimizer.step()
        history[epoch] = loss.item()
        if (epoch+1)%10==0:
            print(f'Epoch {epoch+1}: loss {loss.item():.4f}')
    return model,history


In [ ]:
# @title Grade 7.3 (run, do not edit)
exercise_results['7.3'] = evaluate(globals(), '7.3')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Clear gradients, compute MSE, backpropagate, then take an Adam step.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

In [ ]:
# @title Plot after passing
if exercise_results.get('7.3', False):
    x,y,epochs = mlp_args(2026,80); model,history = train_mlp(x,y,epochs)
    plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.show()


### Exercise 7.4 — `autograd.grad` drills (2025 R2 P1.9) ★★★

[E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** scalar `f(p,q)=p²+q³+pq²` at `(1,2)`: `f_p=6, f_q=16, f_pp=2, f_qq=12, f_pq=4, f_qqq=6`; elementwise `g=x²` via `autograd.grad(torch.sum(g), x, create_graph=True)` → `2x`, then `2`

**Shape contract:** scalar drill output: six scalars; vector drill x: (d,), outputs: two (d,) tensors

**Rules:** Only the starter imports; loops allowed unless a subtask says otherwise.

**Predict first:** Before running, write the input and output shapes for each function. Contract: scalar drill output: six scalars; vector drill x: (d,), outputs: two (d,) tensors.

Correction to the supplied spec: at (1,2), f_qq = 6q+2p = 14. The full tuple is (6,16,2,14,4,6).

**My prediction:** _Write your answer here before coding._

In [ ]:
def scalar_derivatives(p_value,q_value):
    'scalar_derivatives: scalar drill output: six scalars; vector drill x: (d,), outputs: two (d,) tensors. See the exercise task for the required behavior.'
    p = torch.tensor(float(p_value),requires_grad=True)
    q = torch.tensor(float(q_value),requires_grad=True)
    f = p**2+q**3+p*q**2
    fp,fq = torch.autograd.grad(f,(p,q),create_graph=True)
    fpp = torch.autograd.grad(fp,p,create_graph=True)[0]
    fqq = torch.autograd.grad(fq,q,create_graph=True)[0]
    fpq = torch.autograd.grad(fp,q,create_graph=True)[0]
    fqqq = torch.autograd.grad(fqq,q)[0]
    return tuple(v.detach().item() for v in (fp,fq,fpp,fqq,fpq,fqqq))
def vector_derivatives(x):
    'vector_derivatives: scalar drill output: six scalars; vector drill x: (d,), outputs: two (d,) tensors. See the exercise task for the required behavior.'
    x = x.clone().detach().requires_grad_(True)
    g = x**2
    first = torch.autograd.grad(g.sum(),x,create_graph=True)[0]
    second = torch.autograd.grad(first.sum(),x)[0]
    return first.detach(),second.detach()


In [ ]:
# @title Grade 7.4 (run, do not edit)
exercise_results['7.4'] = evaluate(globals(), '7.4')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

create_graph=True keeps derivatives differentiable for the next derivative.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 7.5 — `MyMHA` as `nn.Module` (2025 R2 P2.5) and equality test against 5.6 with copied weights (`nn.Parameter`) ★★★

[C5] [E4] [E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** relative error < 1e-5

**Shape contract:** x: (B,L1,D), y: (B,L2,D); module output: (B,L1,D1)

**Rules:** MyMHA.forward: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: x: (B,L1,D), y: (B,L2,D); module output: (B,L1,D1).

**My prediction:** _Write your answer here before coding._

In [ ]:
class MyMHA(nn.Module):
    def __init__(self,D,D1,H,d):
        '__init__: x: (B,L1,D), y: (B,L2,D); module output: (B,L1,D1). See the exercise task for the required behavior.'
        super().__init__()
        self.H,self.d = H,d
        self.WQ = nn.Parameter(torch.randn(H*d,D)*0.1)
        self.WK = nn.Parameter(torch.randn(H*d,D)*0.1)
        self.WV = nn.Parameter(torch.randn(H*d,D)*0.1)
        self.WO = nn.Parameter(torch.randn(D1,H*d)*0.1)
    def forward(self,x,y):
        'forward: x: (B,L1,D), y: (B,L2,D); module output: (B,L1,D1). See the exercise task for the required behavior.'
        B,L1,D = x.shape
        L2 = y.shape[1]
        H,d = self.H,self.d
        Q = (x@self.WQ.T).reshape(B,L1,H,d).permute(0,2,1,3)
        K = (y@self.WK.T).reshape(B,L2,H,d).permute(0,2,1,3)
        V = (y@self.WV.T).reshape(B,L2,H,d).permute(0,2,1,3)
        scores = Q@K.transpose(-2,-1)/np.sqrt(d)
        weights = torch.softmax(scores,dim=-1)
        return (weights@V).permute(0,2,1,3).reshape(B,L1,H*d)@self.WO.T


In [ ]:
# @title Grade 7.5 (run, do not edit)
exercise_results['7.5'] = evaluate(globals(), '7.5')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Match the NumPy attention steps, using permute and dim=-1.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 7.6 — `Dataset`, `DataLoader`, `collate_fn` with padding and attention masks (2025 R2 P1.4–1.5, P3.8–3.9) ★★★

[E7]

**Source:** 2025 Round 1/2 and 2026 sample/contest, as tagged in the supplied specification. Adapted exercise; source tags supplied with the course specification.

**Task:** `token_id_batch (B,K)`, `attention_mask_batch (B,K)` int64

**Shape contract:** output: token IDs (B,K), attention masks (B,K), both int64

**Rules:** Only the starter imports; loops allowed unless a subtask says otherwise.

**Predict first:** Before running, write the input and output shapes for each function. Contract: output: token IDs (B,K), attention masks (B,K), both int64.

Padding token is 0; masks use 1 for real tokens and 0 for padding, both int64. Loops over sequences are allowed.

**My prediction:** _Write your answer here before coding._

In [ ]:
class TokenDataset(Dataset):
    def __init__(self,sequences):
        '__init__: output: token IDs (B,K), attention masks (B,K), both int64. See the exercise task for the required behavior.'
        self.sequences = sequences
    def __len__(self):
        '__len__: output: token IDs (B,K), attention masks (B,K), both int64. See the exercise task for the required behavior.'
        return len(self.sequences)
    def __getitem__(self,index):
        '__getitem__: output: token IDs (B,K), attention masks (B,K), both int64. See the exercise task for the required behavior.'
        return torch.tensor(self.sequences[index],dtype=torch.int64)
def collate_fn(batch):
    'collate_fn: output: token IDs (B,K), attention masks (B,K), both int64. See the exercise task for the required behavior.'
    lengths = torch.tensor([len(seq) for seq in batch],dtype=torch.int64)
    K = lengths.max().item()
    tokens = torch.zeros(len(batch),K,dtype=torch.int64)
    for i,seq in enumerate(batch):
        tokens[i,:len(seq)] = seq
    mask = (torch.arange(K)[None,:]<lengths[:,None]).to(torch.int64)
    return tokens,mask


In [ ]:
# @title Grade 7.6 (run, do not edit)
exercise_results['7.6'] = evaluate(globals(), '7.6')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Pad to the longest sequence, then compare positions with original lengths.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

## Checklist
- [ ] I can explain and use A4 (see the README knowledge map).
- [ ] I can explain and use A5 (see the README knowledge map).
- [ ] I can explain and use A6 (see the README knowledge map).
- [ ] I can explain and use B1 (see the README knowledge map).
- [ ] I can explain and use B5 (see the README knowledge map).
- [ ] I can explain and use C4 (see the README knowledge map).
- [ ] I can explain and use C5 (see the README knowledge map).
- [ ] I can explain and use E1 (see the README knowledge map).
- [ ] I can explain and use E2 (see the README knowledge map).
- [ ] I can explain and use E4 (see the README knowledge map).
- [ ] I can explain and use E7 (see the README knowledge map).
- [ ] Every exercise says ALL PASS.
- [ ] I reviewed my shape predictions and can explain my mistakes.

Next: [06 mock exam A](06_mock_exam_A.ipynb).